In [1]:
#!pip install statsmodels

In [2]:
import plotly.graph_objects as go
import json
import pandas as pd
import numpy as np
from iso639 import languages
from sklearn.linear_model import LinearRegression
from scipy import stats
from iso639 import languages

In [3]:
base_path = "results"
score_path = "results"
template = "Instruct-Query"
models = [
          "BAAI__bge-m3",
          "codefuse-ai__F2LLM-v2-4B",
          "google__embeddinggemma-300m",
          "intfloat__multilingual-e5-large-instruct",
          "microsoft__harrier-oss-v1-0.6b",
          "Octen__Octen-Embedding-8B",
          "Qwen__Qwen3-Embedding-0.6B",
          "Qwen__Qwen3-Embedding-4B",
          ]
dataset = "mteb__ARCChallenge" 
split = "test"
scores = ["recall@1", "ndcg@10"]
path = lambda model: f"{base_path}/{model}/{dataset}/{split}/{template}_template/"
path_scores = lambda model: f"{score_path}/{model}/{dataset}/{split}/{template}_template/"

In [8]:
# Retrieval prompt

def prompts_retrieval():
    return ["Given a question, retrieve the passage that best answers it.",
            "Retrieve.",
            "Find the most relevant passage that directly answers the question.",
            "Given a question, find a related document.",
            "Retrieve the answer to the question.",
            "Retrieve text based on user query.",
            "Given a question, retrieve Wikipedia passages that answer the question.",
            ]
prompts_appropriate = prompts_retrieval

In [9]:
def construct_df(model, show=False, score= "recall@1", 
            displacement="sim_q2pq", sim_increase="sim_improvement", angulation="chord_similarity",
            columns_to_select=[
                    "score", 
                    "score_distracted", 
                    "score_normalized", 
                    "score_distracted_normalized", 
                    "prompt_text", 
                    "appropriate",
                    "displacement",
                    "sim_improvement",
                    "angulation"
                    ]
                ):
    scores_path= path_scores(model)+f"eval@1_2_5_10.json"
    with open(scores_path) as f:
        scores = json.load(f)
    scores_path2= path_scores(model)+f"eval@1_2_5_10_with_distractors.json"
    with open(scores_path2) as f:
        scores2 = json.load(f)
    df_scores = pd.DataFrame.from_dict(scores).T
    df_scores2 = pd.DataFrame.from_dict(scores2).T
    # there's one prompt twice due to language selection!
    df_scores = df_scores.drop_duplicates(subset="prompt_text")  
    df_scores2 = df_scores2.drop_duplicates(subset="prompt_text")
    # check that the prompt texts match
    # first that they are the same set
    assert set(df_scores["prompt_text"].tolist()) == set(df_scores2["prompt_text"].tolist())
    # and that they're equal length
    assert len(df_scores2) == len(df_scores)
    df_all_scores = df_scores.merge(df_scores2, suffixes=("", "_distracted"), on='prompt_text')
    df = df_all_scores
    df["appropriate"] = [int(p in prompts_appropriate()) for p in df["prompt_text"]]
    df["score"] = [d["mean"] for d in df[score]]
    df["score_distracted"] = [d["mean"] for d in df[f"{score}_distracted"]]
    df["score_normalized"] = stats.zscore(df["score"])
    df["score_distracted_normalized"] = stats.zscore(df["score_distracted"])

    # now read the geometry stuff:
    with open(path(model)+f"prompt_geometry_10nn_1_distractor_and_1_false_positive.json") as f:
        data1 = json.load(f)
    df_geom = pd.DataFrame.from_dict(data1).T
    df_geom = df_geom.drop_duplicates(subset="prompt_text")   # again, one calculated twice!
    for column_name, column in zip(["displacement", "sim_increase", "angulation"],[displacement, sim_increase, angulation]):
        df_geom[column_name] = [d["mean"] for d in df_geom[column]]
    
    # merge ALL
    df = df.merge(df_geom, on='prompt_text')
    if show: display(df.head())
    if columns_to_select:
        return df[columns_to_select]
    return df

In [10]:
dfs = {}
score="recall@1"
for m in models:
    dfs_same_model =[]
    try:
        df = construct_df(m, score=score)
        dfs_same_model.append(df)
    except Exception as e:
        print(f"Cannot construct results for {m}")
        raise(e)
    dfs[m] = pd.concat(dfs_same_model)

In [17]:
import statsmodels.formula.api as smf


def lin_analysis(dfs, score_column, label_column, MV=("score_normalized", "appropriate"), topk=("score_normalized", "appropriate")):

    full_results = {}
    for model_name, df in dfs.items():
        full_results[model_name] = {}
        scores = np.array(df[score_column]).reshape(-1, 1)
        prompt_labels = np.array(df[label_column]).reshape(-1, 1)
        # linear regression
        lreg = LinearRegression().fit(prompt_labels, scores)
        reg_result = lreg.score(prompt_labels, scores)  # r2
        full_results[model_name]["r2"] = reg_result
        if MV:
            full_results[model_name]["r_rb"] = MW_effect_size(df, *MV)
        if topk:
            full_results[model_name]["topk"] = top_k(df, *topk)
    return full_results

def top_k(df, score_column, label_column):
    scores = np.array(df[score_column])
    prompt_labels = np.array(df[label_column])
    appr = scores[prompt_labels == [1]]
    k = len(appr)
    sorted_scores = np.argsort(scores)[::-1][:k]
    best_prompts = np.array([p for p in df["prompt_text"]])[sorted_scores]
    best_prompts_appropriateness = prompt_labels[sorted_scores]
    #print(best_prompts)   # for sanity check
    fraction_of_relevant_in_top = sum(best_prompts_appropriateness)/k
    return fraction_of_relevant_in_top

def MW_effect_size(df, score_column, label_column):
    # top-k and Mann-Whitney-U
    scores = np.array(df[score_column])
    prompt_labels = np.array(df[label_column])
    appr = scores[prompt_labels == [1]]
    not_appr = scores[prompt_labels == [0]]
    stat, p = stats.mannwhitneyu(appr, not_appr, alternative='greater')
    r_rb =  (2 * stat) / (len(appr) * len(not_appr)) -1
    return r_rb
    

full_results = lin_analysis(dfs, "score" , "appropriate")

In [18]:
for model, r2 in full_results.items():
    print(model, r2)

BAAI__bge-m3 {'r2': 0.026981910402722242, 'r_rb': 0.33333333333333326, 'topk': 0.14285714285714285}
codefuse-ai__F2LLM-v2-4B {'r2': 0.2650327249630525, 'r_rb': 0.7925170068027212, 'topk': 0.5714285714285714}
google__embeddinggemma-300m {'r2': 0.0017495241294365194, 'r_rb': -0.04761904761904767, 'topk': 0.14285714285714285}
intfloat__multilingual-e5-large-instruct {'r2': 0.08325282903389086, 'r_rb': 0.5578231292517006, 'topk': 0.2857142857142857}
microsoft__harrier-oss-v1-0.6b {'r2': 0.1012821414719357, 'r_rb': 0.5204081632653061, 'topk': 0.2857142857142857}
Octen__Octen-Embedding-8B {'r2': 0.04998341207085866, 'r_rb': 0.40476190476190466, 'topk': 0.2857142857142857}
Qwen__Qwen3-Embedding-0.6B {'r2': 0.10015202856882066, 'r_rb': 0.5816326530612246, 'topk': 0.42857142857142855}
Qwen__Qwen3-Embedding-4B {'r2': 0.06313079282162648, 'r_rb': 0.44217687074829937, 'topk': 0.14285714285714285}


In [19]:

def to_latex_rows(results):
    for model_name, r in results.items():
        print(" & ".join([model_name, str(np.round(r["topk"],3)), str(np.round(r["r2"],3)), str(np.round(r["r_rb"],3))]), "\\\\")

print(score)
to_latex_rows(full_results)

recall@1
BAAI__bge-m3 & 0.143 & 0.027 & 0.333 \\
codefuse-ai__F2LLM-v2-4B & 0.571 & 0.265 & 0.793 \\
google__embeddinggemma-300m & 0.143 & 0.002 & -0.048 \\
intfloat__multilingual-e5-large-instruct & 0.286 & 0.083 & 0.558 \\
microsoft__harrier-oss-v1-0.6b & 0.286 & 0.101 & 0.52 \\
Octen__Octen-Embedding-8B & 0.286 & 0.05 & 0.405 \\
Qwen__Qwen3-Embedding-0.6B & 0.429 & 0.1 & 0.582 \\
Qwen__Qwen3-Embedding-4B & 0.143 & 0.063 & 0.442 \\


In [ ]:
full_results = mlm_analysis(dfs, "score ~ displacement * appropriate", topk=None)

/users/mynttiam/.local/lib/python3.10/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/users/mynttiam/.local/lib/python3.10/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/users/mynttiam/.local/lib/python3.10/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/users/mynttiam/.local/lib/python3.10/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/users/mynttiam/.local/lib/python3.10/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: T

In [ ]:
full_results

{'BAAI__bge-m3': {'r2': 0.35405487639183547, 'r_rb': 0.4827034883720931},
 'codefuse-ai__F2LLM-v2-4B': {'r2': 0.06787742641136826,
  'r_rb': 0.7369186046511629},
 'google__embeddinggemma-300m': {'r2': 0.16699044686854625,
  'r_rb': 0.22223837209302322},
 'intfloat__multilingual-e5-large-instruct': {'r2': 0.7800381297548623,
  'r_rb': 0.7708575581395349},
 'microsoft__harrier-oss-v1-0.6b': {'r2': 0.5848778478906747,
  'r_rb': 0.8054142441860466},
 'Octen__Octen-Embedding-8B': {'r2': 0.5637806396411087,
  'r_rb': 0.7590843023255813},
 'Qwen__Qwen3-Embedding-0.6B': {'r2': 0.5131052976311653,
  'r_rb': 0.7972383720930232},
 'Qwen__Qwen3-Embedding-4B': {'r2': 0.7885912603853467,
  'r_rb': 0.7712572674418605}}